# Paper — 00b (fixed): NMS Threshold Sweep

Tests whether NMS threshold affects orientation histograms.

**Fix over previous version:** NMS now uses the *bbox shapefile* (YOLO detection
rectangles) for IoU, matching exactly what `SAM2_predict.py` does at inference.
The old version used mask polygon bounds, which caused 80%+ spurious suppression.

**Hypothesis to test:**
- NMS=0.2 → fewer detections, flatter histogram?
- NMS=0.5 → more detections, more biased histogram?

**No GPU needed.** Loads pre-NMS shapefiles and re-applies NMS in memory.

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
from shapely import segmentize
from tqdm import tqdm
from lsnms import nms

from rastertools_BOULDERING import metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

In [ ]:
work_dir        = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster       = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")
prieur_test_dir = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px/preprocessing/test")
gt_tile_ids     = ["1386", "1503", "2054", "2277", "2508"]

# Each model: (dir, mask_pre_nms_glob, bbox_pre_nms_glob)
# bbox_glob=None falls back to mask polygon bounds (SAM2-auto has no bbox file)
MODELS = {
    "YOLOv8":          (work_dir / "exp_yolo_256",           "*-downscaled-mask.shp",  "*-downscaled-bbox.shp"),
    "SAM2 zero-shot":  (work_dir / "exp_sam2_256",           "*-downscaled-mask.shp",  "*-downscaled-bbox.shp"),
    "SAM2 fine-tuned": (work_dir / "exp_sam2_finetuned_256", "*-downscaled-mask.shp",  "*-downscaled-bbox.shp"),
    "SAM2-auto":       (work_dir / "exp_sam2_auto_256",      "*-mask.shp",             None),
}

NMS_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

res             = raster_metadata.get_resolution(in_raster)[0]
AREAL_THRESHOLD = (res ** 2) * (4.74 ** 2)
AR_MIN, AR_MAX  = 1.2, 2.0
BINS            = np.linspace(0, 180, 37)

OUT_DIR = Path("figures_paper"); OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "figure.dpi": 150})
print(f"res={res:.4f} m/px   areal_threshold={AREAL_THRESHOLD:.2f} m²")

In [ ]:
def run_pipeline(poly, res):
    row_seg = pd.Series({"geometry": segmentize(poly, res)})
    try:
        ellipse_poly, _, _, _ = fitEllipse(row_seg)
    except Exception:
        return None
    try:
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
    except Exception:
        return None
    if short_ax < 1e-6:
        return None
    return angle180, long_ax / short_ax


def load_pre_nms(pred_dir, mask_glob, bbox_glob):
    """
    Load pre-NMS mask + bbox shapefiles.

    Key fix: load BOTH files without any area filter first, verify they have
    the same row count, then apply area filter using mask polygon areas and
    apply the SAME index filter to bbox. This ensures mask[i] and bbox[i]
    always correspond to the same detection.
    """
    mask_shps = [p for p in sorted(pred_dir.glob(mask_glob)) if "-nms" not in p.stem]
    if not mask_shps:
        return None, None

    # Load mask — NO area filter yet
    gdfs     = [gpd.read_file(p) for p in mask_shps]
    gdf_mask = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf_mask["poly_area"] = gdf_mask.geometry.area
    if "score" not in gdf_mask.columns:
        gdf_mask["score"] = 1.0
    print(f"  pre-NMS mask (unfiltered): {len(gdf_mask):,}")

    gdf_bbox = None
    if bbox_glob is not None:
        bbox_shps = [p for p in sorted(pred_dir.glob(bbox_glob)) if "-nms" not in p.stem]
        if bbox_shps:
            bdfs     = [gpd.read_file(p) for p in bbox_shps]
            gdf_bbox_raw = gpd.GeoDataFrame(pd.concat(bdfs, ignore_index=True), crs=bdfs[0].crs)
            print(f"  pre-NMS bbox (unfiltered): {len(gdf_bbox_raw):,}")

            if len(gdf_mask) == len(gdf_bbox_raw):
                # Apply area filter using MASK areas, same indices for both
                keep_idx = gdf_mask.index[gdf_mask["poly_area"] >= AREAL_THRESHOLD]
                gdf_mask = gdf_mask.loc[keep_idx].reset_index(drop=True)
                gdf_bbox = gdf_bbox_raw.loc[keep_idx].reset_index(drop=True)
                print(f"  after area filter (mask areas): {len(gdf_mask):,}  [NMS will use bbox geometry]")
            else:
                print(f"  WARNING: mask/bbox row count mismatch before any filtering "
                      f"({len(gdf_mask):,} vs {len(gdf_bbox_raw):,}) — "
                      f"files are from different runs. Falling back to mask bounds.")
                gdf_mask = gdf_mask[gdf_mask["poly_area"] >= AREAL_THRESHOLD].reset_index(drop=True)
                gdf_bbox = None
                print(f"  after area filter: {len(gdf_mask):,}  [NMS will use mask polygon bounds]")
    else:
        gdf_mask = gdf_mask[gdf_mask["poly_area"] >= AREAL_THRESHOLD].reset_index(drop=True)
        print(f"  after area filter: {len(gdf_mask):,}  [NMS will use mask polygon bounds — no bbox file]")

    return gdf_mask, gdf_bbox


def apply_nms(gdf_mask, gdf_bbox, iou_threshold):
    if gdf_bbox is not None:
        boxes = np.array([geom.bounds for geom in gdf_bbox.geometry], dtype=np.float32)
    else:
        boxes = np.array([geom.bounds for geom in gdf_mask.geometry], dtype=np.float32)

    scores = gdf_mask["score"].values.astype(np.float32)
    keep   = nms(boxes=boxes, scores=scores, iou_threshold=iou_threshold,
                 class_ids=None, rtree_leaf_size=32)
    mask = np.zeros(len(gdf_mask), dtype=bool)
    mask[keep] = True
    return mask

In [ ]:
gt_angles = []
for tile_id in gt_tile_ids:
    shp = prieur_test_dir / "labels" / f"M1221383405_{tile_id}_mask.shp"
    if not shp.exists(): continue
    for geom in gpd.read_file(shp).geometry:
        if geom is None or geom.is_empty: continue
        r = run_pipeline(geom, res)
        if r and AR_MIN <= r[1] <= AR_MAX:
            gt_angles.append(r[0])
gt_angles = np.array(gt_angles)
print(f"GT elongated boulders: {len(gt_angles)}")

In [ ]:
# Load pre-NMS files and compute orientations once per model.
# Orientations don't depend on NMS threshold — only computed once.

model_data = {}  # name → {"gdf_mask": ..., "gdf_bbox": ..., "angles": ..., "ars": ...}

for name, (pred_dir, mask_glob, bbox_glob) in MODELS.items():
    print(f"\nLoading {name}...")
    gdf_mask, gdf_bbox = load_pre_nms(pred_dir, mask_glob, bbox_glob)

    if gdf_mask is None:
        print(f"  No pre-NMS files found in {pred_dir} — skipping")
        continue

    nms_src = "bbox shapefile (YOLO boxes)" if gdf_bbox is not None else "mask polygon bounds (fallback)"
    print(f"  {len(gdf_mask)} pre-NMS detections  |  NMS will use: {nms_src}")

    # Verify pre-NMS count is >= post-NMS count (sanity check)
    nms_shps = list(pred_dir.glob(mask_glob.replace(".shp", "-nms.shp")))
    if nms_shps:
        n_post = len(gpd.read_file(nms_shps[0]))
        print(f"  Post-NMS (on disk): {n_post}  {'✓' if len(gdf_mask) >= n_post else '✗ WARNING: pre < post!'}")

    # Compute orientations for ALL pre-NMS detections
    angles, ars = [], []
    for geom in tqdm(gdf_mask.geometry, desc="  orientations", leave=False):
        r = run_pipeline(geom, res)
        if r:
            angles.append(r[0]); ars.append(r[1])
        else:
            angles.append(np.nan); ars.append(np.nan)

    gdf_mask["angle180"]    = angles
    gdf_mask["aspect_ratio"] = ars
    model_data[name] = {"gdf_mask": gdf_mask, "gdf_bbox": gdf_bbox}
    print(f"  orientations done")

In [ ]:
# Apply NMS at each threshold and record counts + elongated-boulder angles
sweep = {}  # name → thresh → {"n_total": int, "angles": np.array}

for name, data in model_data.items():
    gdf_mask = data["gdf_mask"]
    gdf_bbox = data["gdf_bbox"]
    sweep[name] = {}

    for thresh in NMS_THRESHOLDS:
        keep_mask = apply_nms(gdf_mask, gdf_bbox, thresh)
        gdf_keep  = gdf_mask[keep_mask]
        elongated = gdf_keep[
            (gdf_keep["aspect_ratio"] >= AR_MIN) &
            (gdf_keep["aspect_ratio"] <= AR_MAX)
        ]
        sweep[name][thresh] = {
            "n_total": int(keep_mask.sum()),
            "angles":  elongated["angle180"].dropna().values,
        }

    print(f"{name}: " + "  ".join(
        f"NMS={t:.1f}→{sweep[name][t]['n_total']:,}" for t in NMS_THRESHOLDS))

In [ ]:
# Verify: count at the inference threshold should match the on-disk post-NMS count
# SAM2 zero-shot / fine-tuned used 0.5; YOLO used 0.2
inference_thresholds = {
    "YOLOv8":          0.2,
    "SAM2 zero-shot":  0.5,
    "SAM2 fine-tuned": 0.5,
    "SAM2-auto":       0.2,
}

print("Consistency check — sweep count at inference threshold vs on-disk post-NMS count:")
print(f"{'Model':<24}  {'inference thresh':>16}  {'sweep count':>12}  {'on-disk count':>14}  {'match?':>7}")
print("-" * 80)
for name, (pred_dir, mask_glob, _) in MODELS.items():
    if name not in sweep:
        continue
    thresh = inference_thresholds[name]
    sweep_n = sweep[name][thresh]["n_total"]

    nms_glob = mask_glob.replace(".shp", "-nms.shp")
    nms_shps = list(pred_dir.glob(nms_glob))
    disk_n   = len(gpd.read_file(nms_shps[0])) if nms_shps else -1

    match = "✓" if abs(sweep_n - disk_n) < max(10, 0.01 * disk_n) else "✗ MISMATCH"
    print(f"{name:<24}  {thresh:>16.1f}  {sweep_n:>12,}  {disk_n:>14,}  {match:>7}")

In [ ]:
COLORS = {
    "YOLOv8":          "#4C72B0",
    "SAM2 zero-shot":  "tomato",
    "SAM2 fine-tuned": "darkorange",
    "SAM2-auto":       "forestgreen",
}

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, by_thresh in sweep.items():
    counts = [by_thresh[t]["n_total"] for t in NMS_THRESHOLDS]
    ax.plot(NMS_THRESHOLDS, counts, "o-", label=name,
            color=COLORS.get(name, "gray"), lw=1.5, ms=5)

ax.axvline(0.2, color="k", ls="--", lw=0.8, alpha=0.5, label="0.2 (YOLO / original SAM2)")
ax.axvline(0.5, color="k", ls=":",  lw=0.8, alpha=0.5, label="0.5 (current SAM2)")
ax.set_xlabel("NMS IoU threshold")
ax.set_ylabel("Detections (post-NMS)")
ax.set_title("Detection count vs. NMS threshold\n(using YOLO bbox for IoU — matches inference)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_xticks(NMS_THRESHOLDS)
fig.tight_layout()
fig.savefig(OUT_DIR / "nms_counts_fixed.pdf", bbox_inches="tight")
plt.show()

In [ ]:
model_names = list(sweep.keys())
n_thresh    = len(NMS_THRESHOLDS)
n_models    = len(model_names)

fig, axes = plt.subplots(n_thresh, n_models,
                          figsize=(2.8 * n_models, 2.0 * n_thresh),
                          sharex=True)

for row, thresh in enumerate(NMS_THRESHOLDS):
    for col, name in enumerate(model_names):
        ax     = axes[row, col]
        angles = sweep[name][thresh]["angles"]
        n      = sweep[name][thresh]["n_total"]
        color  = COLORS.get(name, "gray")

        counts, _ = np.histogram(angles, bins=BINS)
        cx = (BINS[:-1] + BINS[1:]) / 2
        ax.bar(cx, counts, width=4.5, color=color, edgecolor="white", lw=0.2)

        # GT overlay scaled to match count
        if len(angles) > 0:
            gc, _ = np.histogram(gt_angles, bins=BINS)
            ax.step(cx, gc * len(angles) / max(len(gt_angles), 1),
                    where="mid", color="seagreen", lw=0.9, alpha=0.8)
            ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.5, alpha=0.4)

        ax.set_xlim(0, 180); ax.set_xticks([0, 90, 180])
        ax.spines[["top", "right"]].set_visible(False)

        if row == 0:
            ax.set_title(name, fontsize=8)
        if col == 0:
            ax.set_ylabel(f"NMS={thresh}\n(n={n:,})", fontsize=7)
        if row == n_thresh - 1:
            ax.set_xlabel("angle180 (°)", fontsize=7)

fig.suptitle(
    "Orientation histograms vs. NMS threshold (NMS using YOLO detection boxes — matches inference)\n"
    "Green = GT (scaled). Dashed = uniform. "
    "Key question: does histogram shape change with threshold?",
    fontsize=8, y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / "nms_histograms_fixed.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# KS test vs uniform at each threshold — quantifies how biased each histogram is
from scipy.stats import kstest

print(f"{'Model':<24}  " + "  ".join(f"NMS={t:.1f}" for t in NMS_THRESHOLDS))
print("-" * (24 + 10 * len(NMS_THRESHOLDS)))
for name in sweep:
    row = f"{name:<24}"
    for thresh in NMS_THRESHOLDS:
        angles = sweep[name][thresh]["angles"]
        if len(angles) > 10:
            D, _ = kstest(angles / 180.0, "uniform")
            row += f"  {D:>6.3f}"
        else:
            row += f"  {'—':>6}"
    print(row)
print("\nKS D statistic vs uniform. Lower = flatter = less biased.")
D_gt, _ = kstest(gt_angles / 180.0, "uniform")
print(f"GT reference: D = {D_gt:.3f}")